In [2]:
!pip install stable-baselines3 gymnasium


In [20]:
import gymnasium as gym
import numpy as np
import pandas as pd
from gymnasium import spaces

class MentalHealthEnv(gym.Env):
    def __init__(self, csv_path):
        super().__init__()

        self.df = pd.read_csv(csv_path)
        self.features = [
            "f1_time_spent",
            "f2_distraction",
            "f3_restlessness",
            "f4_anxiety",
            "f5_sleep_irregularity"
        ]

        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(5,), dtype=np.float32
        )

        self.uids = self.df["uid"].unique()
        self.traj = None
        self.idx = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        uid = np.random.choice(self.uids)
        self.traj = self.df[self.df["uid"] == uid].sort_values("day")
        self.idx = 0

        obs = self.traj[self.features].iloc[self.idx].values.astype(np.float32)
        return obs, {}

    def step(self, action):
        prev = self.traj[self.features].iloc[self.idx].values
        self.idx += 1
    
        done = self.idx >= len(self.traj)
    
        if not done:
            nxt = self.traj[self.features].iloc[self.idx].values
        else:
            nxt = prev
    
        risk_prev = prev.mean()
        risk_next = nxt.mean()
    
        # Severity thresholds
        SAFE_T = 0.30
        MOD_T  = 0.50
        SEV_T  = 0.70
    
        reward = 0.0
    
        # ---------------- SAFE ----------------
        if risk_prev < SAFE_T:
            if action == 0:
                reward = +1.0
            else:
                reward = -1.0
    
        # ------------- MODERATE ---------------
        elif risk_prev < MOD_T:
            if action == 1:
                reward = +1.0
            elif action == 2:
                reward = +0.3    # tolerable escalation
            else:
                reward = -0.7
    
        # ------------ WORSENING ---------------
        elif risk_prev < SEV_T:
            if action == 2:
                reward = +1.0
            elif action == 3:
                reward = +0.4
            else:
                reward = -1.0
    
        # ------------- SEVERE -----------------
        else:
            if action == 3:
                reward = +1.0
            elif action == 2:
                reward = -0.3    # underreaction
            else:
                reward = -1.0
    
        return nxt.astype(np.float32), reward, done, False, {}
    
    


In [29]:
env = MentalHealthEnv("temporal_smmh_rl.csv")
obs, _ = env.reset()
print(obs, obs.shape)


[0.39226407 0.24776785 0.         0.97812605 0.7231222 ] (5,)


In [32]:
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv

env = DummyVecEnv([lambda: MentalHealthEnv("temporal_smmh_rl.csv")])

model = DQN(
    policy="MlpPolicy",
    env=env,
    learning_rate=1e-3,
    buffer_size=50000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    target_update_interval=500,
    exploration_fraction=0.3,
    exploration_final_eps=0.05,
    verbose=1
)

model.learn(total_timesteps=30000)

model.save("dqn_mental_health finall")


Using cpu device
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.998    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 1108     |
|    time_elapsed     | 0        |
|    total_timesteps  | 20       |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.996    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1200     |
|    time_elapsed     | 0        |
|    total_timesteps  | 40       |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.994    |
| time/               |          |
|    episodes         | 12       |
|    fps              | 1113     |
|    time_elapsed     | 0        |
|    total_timesteps  | 60       |
----------------------------------
----------------------------------
| r

In [33]:
obs = env.reset()
total_reward = 0

for _ in range(20):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = env.step(action)
    total_reward += reward
    if done:
        obs = env.reset()

print("Total test reward:", total_reward)


Total test reward: [19.3]


In [34]:
import numpy as np

safe_user = np.array([
    0.20,  # f1_time_spent
    0.15,  # f2_distraction
    0.10,  # f3_restlessness
    0.12,  # f4_anxiety
    0.08   # f5_sleep_irregularity
], dtype=np.float32)

action, _ = model.predict(safe_user, deterministic=True)

action_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

print("Predicted Action:", int(action))
print("Predicted Risk Level:", action_map[int(action)])


Predicted Action: 0
Predicted Risk Level: Good


In [35]:
import numpy as np

# Sample user state
sample_user = np.array([
    0.65,  # f1_time_spent
    0.55,  # f2_distraction
    0.50,  # f3_restlessness
    0.60,  # f4_anxiety
    0.70   # f5_sleep_irregularity
], dtype=np.float32)

# Predict action
action, _ = model.predict(sample_user, deterministic=True)

print("Predicted Action:", int(action))

# Map to meaning
action_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

print("Predicted Mental Health Risk:", action_map[int(action)])


Predicted Action: 2
Predicted Mental Health Risk: Worsening


In [36]:
import numpy as np

samples = {
    "Good":       [0.20, 0.15, 0.10, 0.12, 0.08],
    "Moderate":   [0.40, 0.35, 0.30, 0.38, 0.32],
    "Worsening":  [0.60, 0.55, 0.50, 0.58, 0.62],
    "Severe":     [0.85, 0.80, 0.78, 0.82, 0.90]
}

action_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

for name, state in samples.items():
    obs = np.array(state, dtype=np.float32)
    action, _ = model.predict(obs, deterministic=True)
    print(f"Input: {name:9s} → Predicted: {action_map[int(action)]}")


Input: Good      → Predicted: Good
Input: Moderate  → Predicted: Moderate
Input: Worsening → Predicted: Worsening
Input: Severe    → Predicted: Severe


In [37]:
import numpy as np

borderline_user = np.array([
    0.52,  # f1_time_spent
    0.48,  # f2_distraction
    0.46,  # f3_restlessness
    0.50,  # f4_anxiety
    0.49   # f5_sleep_irregularity
], dtype=np.float32)

action, _ = model.predict(borderline_user, deterministic=True)

action_map = {
    0: "Good",
    1: "Moderate",
    2: "Worsening",
    3: "Severe"
}

print("Predicted Action:", int(action))
print("Predicted Risk Level:", action_map[int(action)])


Predicted Action: 2
Predicted Risk Level: Worsening


In [38]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("temporal_smmh_rl.csv")

features = [
    "f1_time_spent",
    "f2_distraction",
    "f3_restlessness",
    "f4_anxiety",
    "f5_sleep_irregularity"
]

def risk_to_label(r):
    if r < 0.30:
        return 0
    elif r < 0.50:
        return 1
    elif r < 0.70:
        return 2
    else:
        return 3

y_true, y_pred = [], []

for _, row in df.iterrows():
    obs = row[features].values.astype(np.float32)
    risk = obs.mean()
    
    y_true.append(risk_to_label(risk))
    action, _ = model.predict(obs, deterministic=True)
    y_pred.append(int(action))

acc = accuracy_score(y_true, y_pred)
print("Policy Agreement Accuracy:", acc)

print(classification_report(
    y_true, y_pred,
    target_names=["Good", "Moderate", "Worsening", "Severe"]
))


Policy Agreement Accuracy: 0.9194630872483222
              precision    recall  f1-score   support

        Good       0.99      0.99      0.99       226
    Moderate       0.99      0.86      0.92       422
   Worsening       0.90      0.89      0.90       560
      Severe       0.83      1.00      0.91       282

    accuracy                           0.92      1490
   macro avg       0.93      0.94      0.93      1490
weighted avg       0.93      0.92      0.92      1490



In [39]:
model.save("dqn_mental_health_final")
print("Model saved successfully")


Model saved successfully
